# **Data Analysis II**
## *Lab 6: Likelihood Fitting*

In real HEP analyses, binned least-squares / $\chi^2$ minimization have two major limitations:
 1. Resonances like $Z \to \mu^+\mu^-$ have an intrinsic Breit–Wigner width convolved with the Guassian detector resolution, sitting on top of a falling exponential background.
 2. In low-count channels like $H \to 4\ell$, bins often contain $0, 1, 2$ events, where Gaussian least-squares errors ($1/\sqrt{N}$) break down completely.

Today we will take a look at **likelihood fitting** using **`zfit`**, a `python` replacement for `RooFit`. We will split the lab into two parts:
* **Part 1 ($Z \to \mu^+\mu^-$):** Compare `curve_fit` and unbinned `zfit` fits on high-statistics data, build a composite **(Cauchy $\otimes$ Gaussian) + Exponential** extended PDF, compute fit uncertainties, and inspect pull distributions.
* **Part 2 ($H \to ZZ^* \to 4\ell$):** Take the cut-based and BDT selected events, see why binned $\chi^2$ fails in the sparse regime and perform an **unbinned extended likelihood fit** to extract $m_H$ and $N_{\text{sig}}$.


In [ ]:
import atlasopenmagic as atom
import awkward as ak
import uproot
import numpy as np
import scipy.optimize as opt
import matplotlib.pyplot as plt
import mplhep as mh
import vector
import zfit
from itertools import combinations
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

In [ ]:
mh.style.use("ATLAS")
atom.set_release("2025e-13tev-beta")
lumi = 36600  # pb^-1 

Let's define a reusable plotting helper. We pass this function:
 - `values`: unbinned observable values (invariant masses `m_mumu` or `m4l_data`) to use as the data points.
 - `bins`: Bin edges `[x_min, ..., x_max]` used to histogram `values`.
 - `components`: a list of dictionaries describing fit components to draw and sum into the total curve:
  * `"pdf"`: normalized `zfit` PDF.
  * `"yield"`: fitted event yield $N_{\text{component}}$ scaling that PDF.
  * `"label"`: legend label string.
  * `"color"` *(optional, default `"tab:blue"`)* and `"linestyle"` *(optional, default `"--"`)*: matplotlib styling.
* `xlabel`: label for the shared x-axis (e.g. `r"$m_{\mu\mu}$ [GeV]"`).
* `title` (*optional*): litle placed above the main plot panel.
* `total_label` (*optional*): legend label for the summed signal + background curve.


In [ ]:
def plot_zfit_result(
    values,
    bins,
    components,
    xlabel,
    title=None,
    total_label="Total fit",
):
    
    # Histogram the data
    bin_centers = 0.5 * (bins[1:] + bins[:-1])
    bin_width = bins[1] - bins[0]
    counts, _ = np.histogram(values, bins=bins)
    counts_err = np.sqrt(counts)

    x_grid = np.linspace(bins[0], bins[-1], 600)
    total_curve = np.zeros_like(x_grid)
    total_at_centers = np.zeros_like(bin_centers)

    fig, (ax, ax_pull) = plt.subplots(
        2, 1, figsize=(9, 7), sharex=True,
        gridspec_kw={"height_ratios": [3.2, 1.0], "hspace": 0.08},
    )

    ax.errorbar(
        bin_centers, counts, yerr=counts_err,
        fmt="ko", markersize=4, label="Data", zorder=5,
    )

    # Take each component PDF and plot it
    for comp in components:
        pdf_grid = np.asarray(comp["pdf"].pdf(x_grid))
        pdf_centers = np.asarray(comp["pdf"].pdf(bin_centers))
        y_grid = comp["yield"] * pdf_grid * bin_width
        y_centers = comp["yield"] * pdf_centers * bin_width
        total_curve += y_grid
        total_at_centers += y_centers
        ax.plot(
            x_grid, y_grid,
            linestyle=comp.get("linestyle", "--"),
            color=comp.get("color", "tab:blue"),
            linewidth=1.8,
            label=comp["label"],
        )

    # Plot the total fit
    ax.plot(x_grid, total_curve, color="crimson", linewidth=2.2, label=total_label, zorder=4)
    ax.set_ylabel(f"Events / {bin_width:.2f} GeV")
    ax.set_ylim(bottom=0)
    if title:
        ax.set_title(title)
    ax.legend()

    # Make the pull plot
    pulls = (counts - total_at_centers) / np.sqrt(np.maximum(total_at_centers, 1e-9))
    ax_pull.axhline(0.0, color="crimson", linewidth=1.2)
    ax_pull.axhspan(-1.0, 1.0, color="gold", alpha=0.25)
    ax_pull.axhspan(-3.0, 3.0, color="lightgreen", alpha=0.15)
    ax_pull.errorbar(bin_centers, pulls, yerr=np.ones_like(pulls), fmt="ko", markersize=3.5)
    ax_pull.set_ylim(-4.5, 4.5)
    ax_pull.set_ylabel("Pull")
    ax_pull.set_xlabel(xlabel)

    plt.show()

## **Part 1: Precision Fitting on $Z \to \mu^+\mu^-$**

We start with the familiar $Z \to \mu^+\mu^-$ channel. We benefit from the fact that the `2muons` skim has huge statistics, and we have millions of dimuon pairs around $m_Z \approx 91.2\text{ GeV}$, so we avoid low-yield pathologies.

In [ ]:
# We will look only at data, no MC!
defs_z = {
    "Data": {"dids": ["data"]},
}
samples_z = atom.build_dataset(defs_z, skim="2muons", protocol="root")

variables_z = [
    "trigM", "lep_n", "lep_pt", "lep_eta", "lep_phi", "lep_e",
    "lep_charge", "lep_type", "lep_isMediumID", "lep_isLooseIso",
    "lep_isTrigMatched",
]

In [ ]:
def calc_inv_mass(lep_pt, lep_eta, lep_phi, lep_e):
    p4 = vector.zip({
        "pt": lep_pt,
        "eta": lep_eta,
        "phi": lep_phi,
        "E": lep_e,
    })
    return ak.sum(p4, axis=1).M

We define a function that parses the dataset and applies all of the cuts we are used to, also computing the invariant mass of the $\mu^+\mu^-$ pair.

In [ ]:
def select_dimuon_data(file_list, fraction=0.1):
    chunks = []
    for filename in file_list:
        tree = uproot.open(filename + ":analysis")
        stop = max(1, int(fraction * tree.num_entries))
        for data in tree.iterate(
            filter_name=variables_z, library="ak", entry_stop=stop, step_size=100_000
        ):
            mask = (
                data["trigM"]
                & (ak.sum(data["lep_isTrigMatched"], axis=1) >= 1)
                & (data["lep_n"] == 2)
                & (ak.sum(data["lep_charge"], axis=1) == 0)
                & ak.all(data["lep_type"] == 13, axis=1)
                & ak.all(data["lep_isMediumID"] & data["lep_isLooseIso"], axis=1)
                & ak.all(data["lep_pt"] > 20.0, axis=1)
            )
            selected = data[mask]
            if len(selected) > 0:
                chunks.append(
                    calc_inv_mass(
                        selected["lep_pt"],
                        selected["lep_eta"],
                        selected["lep_phi"],
                        selected["lep_e"],
                    )
                )
    return np.asarray(ak.to_numpy(ak.concatenate(chunks)))

We take the limits of the fit in the $m_{\mu\mu}\in [70, 110] \text{ GeV}$ range.

In [ ]:
z_fit_low, z_fit_high = 70.0, 110.0
m_mumu_all = select_dimuon_data(samples_z["Data"]["list"], fraction=0.1)
m_mumu = m_mumu_all[(m_mumu_all >= z_fit_low) & (m_mumu_all <= z_fit_high)]
print(f"Selected dimuon events in [{z_fit_low:.0f}, {z_fit_high:.0f}] GeV: {len(m_mumu)}")

### *Why a binned Gaussian is not enough*

Let's repeat the `curve_fit` Gaussian fit in the $[70, 110] \text{ GeV}$ range and compute the reduced $\chi^2 / \text{ndf}$.

In [ ]:
bins_z = np.linspace(z_fit_low, z_fit_high, 81)  
centers_z = 0.5 * (bins_z[1:] + bins_z[:-1])
counts_z, _ = np.histogram(m_mumu, bins=bins_z)
err_z = np.sqrt(np.maximum(counts_z, 1))

# Define the Gaussian with which to fit
def gauss_model(x, amp, mu, sigma):
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

# Perform the fit 
popt, pcov = opt.curve_fit(
    gauss_model, centers_z, counts_z,
    p0=[counts_z.max(), 91.2, 2.5], sigma=err_z, absolute_sigma=True,
)
# Extract the errors and compute chi2 / ndf
perr = np.sqrt(np.diag(pcov))
pred_gauss = gauss_model(centers_z, *popt)
chi2_ndf = np.sum(((counts_z - pred_gauss) / err_z) ** 2) / (len(centers_z) - len(popt))

# We take a look at the parameters 
print(f"curve_fit Gaussian: mu = {popt[1]:.4f} +/- {perr[1]:.4f} GeV, "
      f"sigma = {popt[2]:.4f} +/- {perr[2]:.4f} GeV, chi2/ndf = {chi2_ndf:.2f}")

# Plot the results
fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(centers_z, counts_z, yerr=np.sqrt(counts_z), fmt="ko", markersize=3.5, label="Data")
x_fine = np.linspace(z_fit_low, z_fit_high, 500)
ax.plot(x_fine, gauss_model(x_fine, *popt), color="crimson", lw=2,
        label=rf"curve_fit Gauss ($\chi^2/\mathrm{{ndf}} = {chi2_ndf:.1f}$)")
ax.set_xlabel(r"$m_{\mu\mu}$ [GeV]")
ax.set_ylabel(f"Events / {(bins_z[1] - bins_z[0]):.2f} GeV")
ax.legend()
plt.tight_layout()
plt.show()

Notice how a pure Gaussian misses the Breit–Wigner resonance tails in the sidebands. The exponential background is there, but the contamination is very small.

### *The first unbinned `zfit` fit (Gaussian)*

In `zfit`, building a fit always follows the same 4 steps:
1. **Observable space**: defining the variable and its limits.
2. **Parameters & PDF**: the underlying PDF and its parameters.
3. **Data & Loss**: the data to fit and the loss function.
4. **Minimizer**: algorithm that will minimize the loss function.

Let's start with a single Gaussian PDF so you can see the syntax cleanly.

In [ ]:
# The variable of the distribution
obs_z = zfit.Space("m_mumu", limits=(z_fit_low, z_fit_high))

# The data 
data_z = zfit.Data.from_numpy(obs=obs_z, array=m_mumu)

# The parameters of the PDF
mu_z_g = zfit.Parameter("mu_z_g", 91.2, 85.0, 97.0)
sigma_z_g = zfit.Parameter("sigma_z_g", 2.5, 0.5, 8.0)
# The underlying Gaussian PDF 
pdf_z_gauss = zfit.pdf.Gauss(mu=mu_z_g, sigma=sigma_z_g, obs=obs_z)

# The loss function and the minimizer
nll_z_gauss = zfit.loss.UnbinnedNLL(model=pdf_z_gauss, data=data_z)
minimizer = zfit.minimize.Minuit()

# The fitting itself and uncertainty computation
result_z_gauss = minimizer.minimize(nll_z_gauss)
result_z_gauss.hesse()
print(result_z_gauss)

We obtained some results for the $Z$ boson mass and its width!
 - $m_Z = 90.2063 \pm 0.0032 \text{ GeV}$
 - $\sigma_Z = 4.9562 \pm 0.0023 \text{ GeV}$ 

Now, let's take a look at:

### *Signal + Background Model: (Cauchy $\otimes$ Gaussian) + Exponential*

We will build an **extended maximum likelihood** model:
* **Signal:** the convolution of a relativistic Breit–Wigner resonance with Gaussian detector resolution. This function is also known as a Voigt function.
  > **Note on `gamma`:** In `zfit.pdf.Voigt`, `gamma` is the **half-width at half-maximum (HWHM)** of the Breit-Wigner component, so the full decay width is $\Gamma_Z = 2\gamma$. (PDG: $m_Z = 91.1876\text{ GeV}$, $\Gamma_Z = 2.4955\text{ GeV}$)
* **Background:** an exponential capturing the falling continuum.
* **Extended yields:** we attach `n_sig_z` and `n_bkg_z` via `.create_extended(...)`, combine them with `zfit.pdf.SumPDF`, and minimize `zfit.loss.ExtendedUnbinnedNLL`.

We take a look at both covariance and minimizer errors.

In [ ]:
# The Z boson mass
m_z_v = zfit.Parameter("m_z_v", 91.2, 88.0, 94.0)
# The parameters of the gaussian and the lorentzian
sigma_z_v = zfit.Parameter("sigma_z_v", 1.8, 0.3, 5.0)
gamma_z_v = zfit.Parameter("gamma_z_v", 1.25, 0.2, 4.0)
# The normalization constant for the signal
sig_z_pdf = zfit.pdf.Voigt(m=m_z_v, sigma=sigma_z_v, gamma=gamma_z_v, obs=obs_z)

# Decay constant for the exponential
lam_z = zfit.Parameter("lam_z", -0.04, -0.5, 0.1)
# Normalization constant for the exponential
bkg_z_pdf = zfit.pdf.Exponential(lam_z, obs=obs_z)

n_total_z = float(len(m_mumu))
n_sig_z = zfit.Parameter("n_sig_z", 0.90 * n_total_z, 0.0, 1.5 * n_total_z)
n_bkg_z = zfit.Parameter("n_bkg_z", 0.10 * n_total_z, 0.0, 1.5 * n_total_z)

# We create the signal + background model
model_z = zfit.pdf.SumPDF([
    sig_z_pdf.create_extended(n_sig_z),
    bkg_z_pdf.create_extended(n_bkg_z),
])

# We fit the likelihood function and extract the uncertainties
nll_z = zfit.loss.ExtendedUnbinnedNLL(model=model_z, data=data_z)
result_z = minimizer.minimize(nll_z)
hesse_z = result_z.hesse()
minos_z, _ = result_z.errors()
print(result_z)

m_z_val = result_z.params[m_z_v]["value"]
m_z_err_lo = minos_z[m_z_v]["lower"]
m_z_err_hi = minos_z[m_z_v]["upper"]
gamma_val = result_z.params[gamma_z_v]["value"]
gamma_err = hesse_z[gamma_z_v]["error"]

# We take a look at the results
print(f"\nExtracted Z mass:  m_Z = {m_z_val:.4f} ({m_z_err_lo:+.4f}, {m_z_err_hi:+.4f}) GeV  [PDG: 91.1876 GeV]")
print(f"Extracted Z width: Gamma_Z = 2*gamma = {2 * gamma_val:.4f} +/- {2 * gamma_err:.4f} GeV  [PDG: 2.4955 GeV]")
print(f"Detector resolution: sigma = {result_z.params[sigma_z_v]['value']:.4f} +/- {hesse_z[sigma_z_v]['error']:.4f} GeV")

# We plot the final fitted function
plot_zfit_result(
    values=m_mumu,
    bins=bins_z,
    components=[
        {
            "pdf": sig_z_pdf,
            "yield": result_z.params[n_sig_z]["value"],
            "color": "tab:blue",
            "linestyle": "--",
            "label": "Voigt signal",
        },
        {
            "pdf": bkg_z_pdf,
            "yield": result_z.params[n_bkg_z]["value"],
            "color": "tab:orange",
            "linestyle": ":",
            "label": "Exponential bkg",
        },
    ],
    xlabel=r"$m_{\mu\mu}$ [GeV]",
    title=r"Extended Unbinned ML Fit: $Z \to \mu^+\mu^-$",
)

We can clearly see that this physical likelihood function fits the data points much better, accounting for the Cauchy-Lorentz tails and also the exponential background. The pull plot also shows us how much the observed data deviates from the fitted function in each bin.


## *Part 2: Low-Statistics Unbinned Fit on $H \to ZZ^* \to 4\ell$*

Now let's move to the $H \to ZZ^* \to 4\ell$. Here, the total number of events in the Higgs mass region is very small (tens of events in $36.6\text{ fb}^{-1}$ of data).

We will:
 1. Reuse the Lab 4 pre-selection (`skim='exactly4lep'`, `fraction = 1.0`)and binary BDT selection.
 2. Focus on the fit window **`105–160 GeV`**, which avoids the sharp $Z \to 4\ell$ peak near $m_Z$ and leaves a smooth falling $ZZ^*$ (+ reducible) background beneath the $125\text{ GeV}$ Higgs peak.
 3. Demonstrate why `curve_fit` fails on sparse bins.
 4. Calibrate the Higgs signal width $\sigma_H$ and background slope $\lambda_{\text{bkg}}$ from MC, then run an **unbinned extended maximum likelihood fit** on real data.


In [ ]:
# The data samples we will use
defs_h = {
    "Data": {"dids": ["data"]},
    r"Background $Z,t\bar{t},t\bar{t}+V,VVV$": {
        "dids": [
            364245, 364247, 410408, 410470, 410155, 410218, 410219, 410220, 412043,
            364242, 364243, 364246, 364248, 700320, 700321, 700322, 700323, 700324,
            700325, 700587, 700792, 700793, 700794,
        ]
    },
    r"Background $ZZ^{*}$": {"dids": [700600, 700601]},
    r"Signal ($m_H$ = 125 GeV)": {
        "dids": [345060, 346228, 346340, 346341, 346342, 345066, 346414, 346511]
    },
}

data_samples_h = atom.build_dataset(defs_h, skim="exactly4lep", protocol="root")

In [ ]:
# The variables we will use
variables_h = [
    "mcWeight", "ScaleFactor_LepTRIGGER", "trigE", "trigM", "lep_n", "lep_pt",
    "lep_eta", "lep_phi", "lep_e", "lep_charge", "lep_type", "met", "met_phi",
    "lep_isMediumID", "lep_isLooseIso", "sum_of_weights", "xsec",
    "lep_isTrigMatched", "ScaleFactor_ELE", "ScaleFactor_MUON",
    "ScaleFactor_PILEUP", "filteff", "kfac", "lep_isLooseID",
]

In [ ]:
# Function computing MC weights
def calc_weight_4l(data):
    return (
        data["ScaleFactor_ELE"] * data["ScaleFactor_MUON"]
        * data["ScaleFactor_LepTRIGGER"] * data["ScaleFactor_PILEUP"]
        * (data["mcWeight"] / data["sum_of_weights"])
        * (data["xsec"] * data["filteff"] * data["kfac"] * lumi)
    )

In [ ]:
# Function computing 4-lepton invariant mass
def calc_mass_4l(lep_pt, lep_eta, lep_phi, lep_e):
    p4 = vector.zip(
        {"pt": lep_pt, 
        "eta": lep_eta, 
        "phi": lep_phi, 
        "E": lep_e})
    return ak.sum(p4, axis=1).M

In [ ]:
# Function that processes the data samples and applies the cuts we have seen last times
def process_4l_file(filename, sample_name, fraction=1.0):
    tree = uproot.open(filename + ":analysis")
    chunks = []
    stop = max(1, int(fraction * tree.num_entries))
    for data in tree.iterate(
        filter_name=variables_h, library="ak", entry_stop=stop, step_size=100_000
    ):
        sum_type = ak.sum(data["lep_type"], axis=1)
        lep_type_ok = (sum_type == 44) | (sum_type == 48) | (sum_type == 52)
        id_iso_ok = (
            ak.sum(
                ((data["lep_type"] == 13) & data["lep_isMediumID"] & data["lep_isLooseIso"])
                | ((data["lep_type"] == 11) & data["lep_isLooseID"] & data["lep_isLooseIso"]),
                axis=1,
            ) == 4
        )
        mask = (
            lep_type_ok
            & (ak.sum(data["lep_charge"], axis=1) == 0)
            & (data["trigE"] | data["trigM"])
            & (ak.sum(data["lep_isTrigMatched"], axis=1) >= 1)
            & id_iso_ok
        )
        data = data[mask]
        if len(data) == 0:
            continue
        data["inv_mass"] = calc_mass_4l(
            data["lep_pt"], data["lep_eta"], data["lep_phi"], data["lep_e"]
        )
        data["Weight"] = (
            calc_weight_4l(data) if "Data" not in sample_name else ak.ones_like(data["met"])
        )
        chunks.append(data)
    return ak.concatenate(chunks, axis=0) if chunks else ak.Array([])

In [ ]:
# We process the data samples
data_h = {}
for sample in data_samples_h:
    print(f"Processing {sample}...")
    frames = [process_4l_file(f, sample, fraction=1.0) for f in data_samples_h[sample]["list"]]
    frames = [f for f in frames if len(f) > 0]
    data_h[sample] = ak.concatenate(frames, axis=0)

### *Cut-based baseline*

Before applying any machine learning, let's look at how the $H \to ZZ^* \to 4\ell$ invariant mass spectrum looks using only the cuts applied above:
1. On the full $[80, 250] \text{ GeV}$ range, notice the large $Z \to 4\ell$ peak near $91.2\text{ GeV}$ and the $ZZ^*$ + reducible ($Z, t\bar{t}$) background burying the small $125\text{ GeV}$ Higgs peak.
2. In the $[105, 160] \text{ GeV}$ fit window, we can also run our unbinned `zfit` model **before** the BDT cut, so we have a direct cut-based baseline to compare against the post-BDT fit later!

In [ ]:
# We define our fit limits and our variable for the fitting
h_fit_low, h_fit_high = 105.0, 160.0
obs_h = zfit.Space("m_4l", limits=(h_fit_low, h_fit_high))
bins_h = np.linspace(h_fit_low, h_fit_high, 23)

In [ ]:
# Helper function plotting stacked MC vs Data over [80, 250] GeV and zoomed in over [105, 160] GeV 
def plot_mc_vs_data_4l(m_red, w_red, m_zz, w_zz, m_sig, w_sig, m_dat, title):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, bins, subtitle in [
        (axes[0], np.arange(80, 252.5, 2.5), "Full spectrum (80–250 GeV)"),
        (axes[1], bins_h, "Fit window (105–160 GeV)"),
    ]:
        centers = 0.5 * (bins[1:] + bins[:-1])
        counts, _ = np.histogram(m_dat, bins=bins)
        ax.hist(
            [m_red, m_zz, m_sig],
            bins=bins,
            weights=[w_red, w_zz, w_sig],
            stacked=True,
            color=["purple", "red", "cyan"],
            label=[r"$Z,t\bar{t},t\bar{t}+V,VVV$", r"$ZZ^{*}$", r"Higgs ($m_H = 125$ GeV)"],
        )
        ax.errorbar(centers, counts, yerr=np.sqrt(counts), fmt="ko", markersize=3.5, label="Data")
        ax.set_xlabel(r"$m_{4\ell}$ [GeV]")
        ax.set_ylabel(f"Events / {(bins[1] - bins[0]):.1f} GeV")
        ax.set_title(subtitle)
        ax.legend(fontsize=9)
    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    plt.show()

We define a helper function that estimates the parameters of the distributions from the MC samples and then performs an unbinned likelihood fit on the data. We pass this function:
 - `m_sig_mc`: the 4-lepton invariant masses of the signal MC events, used to obtain the detector resolution $\sigma_H$.
 - `w_sig_mc`: the corresponding event weights.
 - `m_bkg_mc`: the 4-lepton invariant masses of the background MC events, used to obtain the decay constant of the exponential bacgkround $\lambda_{\text{bkg}}$.
 - `w_bkg_mc`: the corresponding event weights.
 - `m_data`: the unweighted 4-lepton invariant masses from real data, on which we will perform the fit.
 - `prefix`: unique tag for the parameters to avoid conflicts.
 - `title`: plotting title.

In [ ]:
# We calibrate the signal and background shapes from MC and then perform the fit on data
def fit_higgs_zfit(m_sig_mc, w_sig_mc, m_bkg_mc, w_bkg_mc, m_data, prefix, title):
    # Signal MC shape: detector resolution sigma_H
    pos_sig = w_sig_mc > 0
    data_sig_mc = zfit.Data.from_numpy(
        obs=obs_h, array=m_sig_mc[pos_sig], weights=w_sig_mc[pos_sig]
    )
    mu_mc = zfit.Parameter(f"{prefix}_mu_mc", 125.0, 120.0, 130.0)
    sigma_mc = zfit.Parameter(f"{prefix}_sigma_mc", 1.8, 0.4, 5.0)
    pdf_sig_mc = zfit.pdf.Gauss(mu=mu_mc, sigma=sigma_mc, obs=obs_h)
    res_sig_mc = minimizer.minimize(zfit.loss.UnbinnedNLL(model=pdf_sig_mc, data=data_sig_mc))

    # Background MC shape: decay constant lambda_bkg
    pos_bkg = w_bkg_mc > 0
    data_bkg_mc = zfit.Data.from_numpy(
        obs=obs_h, array=m_bkg_mc[pos_bkg], weights=w_bkg_mc[pos_bkg]
    )
    lam_mc = zfit.Parameter(f"{prefix}_lam_mc", -0.02, -0.2, 0.05)
    pdf_bkg_mc = zfit.pdf.Exponential(lam_mc, obs=obs_h)
    res_bkg_mc = minimizer.minimize(zfit.loss.UnbinnedNLL(model=pdf_bkg_mc, data=data_bkg_mc))

    sigma_fixed = float(res_sig_mc.params[sigma_mc]["value"])
    lam_fixed = float(res_bkg_mc.params[lam_mc]["value"])
    n_sig_sm = float(np.sum(w_sig_mc))
    n_bkg_mc = float(np.sum(w_bkg_mc))

    # We now define the parameters for the fit on real data
    data_zfit = zfit.Data.from_numpy(obs=obs_h, array=m_data)
    m_h = zfit.Parameter(f"{prefix}_m_h", 125.0, 118.0, 132.0)
    sigma_h = zfit.Parameter(f"{prefix}_sigma_h", sigma_fixed, 0.4, 5.0)
    sigma_h.floating = False
    sig_pdf = zfit.pdf.Gauss(mu=m_h, sigma=sigma_h, obs=obs_h)

    lam_h = zfit.Parameter(f"{prefix}_lam_h", lam_fixed, -0.2, 0.05)
    lam_h.floating = False
    bkg_pdf = zfit.pdf.Exponential(lam_h, obs=obs_h)

    n_tot = float(len(m_data))
    n_sig = zfit.Parameter(f"{prefix}_n_sig", max(1.0, n_sig_sm), 0.0, 2.0 * n_tot)
    n_bkg = zfit.Parameter(f"{prefix}_n_bkg", max(1.0, n_bkg_mc), 0.0, 2.0 * n_tot)

    # We build the total s+b likelihood and perform the fit 
    model = zfit.pdf.SumPDF([
        sig_pdf.create_extended(n_sig),
        bkg_pdf.create_extended(n_bkg),
    ])
    result = minimizer.minimize(zfit.loss.ExtendedUnbinnedNLL(model=model, data=data_zfit))
    hesse = result.hesse()
    minos, _ = result.errors()
    print(result)

    m_h_val = float(result.params[m_h]["value"])
    n_sig_val = float(result.params[n_sig]["value"])
    n_sig_err = float(hesse[n_sig]["error"])
    n_bkg_val = float(result.params[n_bkg]["value"])
    mu_hat = n_sig_val / n_sig_sm
    mu_hat_err = n_sig_err / n_sig_sm

    print(f"\nMC shape calibration -> sigma_H = {sigma_fixed:.2f} GeV, lambda_bkg = {lam_fixed:.4f} GeV^-1")
    print(f"Measured m_H = {m_h_val:.2f} ({minos[m_h]['lower']:+.2f}, {minos[m_h]['upper']:+.2f}) GeV  [PDG: 125.20 GeV]")
    print(f"Fitted N_sig = {n_sig_val:.2f} +/- {n_sig_err:.2f}  (SM MC expectation: {n_sig_sm:.2f})")
    print(f"Implied signal strength mu_hat = {mu_hat:.2f} +/- {mu_hat_err:.2f}")

    # We plot the results to take a look at them
    plot_zfit_result(
        values=m_data,
        bins=bins_h,
        components=[
            {"pdf": sig_pdf, "yield": n_sig_val, "color": "tab:cyan", "linestyle": "--", "label": r"Higgs signal ($m_H$ fit)"},
            {"pdf": bkg_pdf, "yield": n_bkg_val, "color": "tab:red", "linestyle": ":", "label": r"$ZZ^*$ + reducible bkg"},
        ],
        xlabel=r"$m_{4\ell}$ [GeV]",
        title=title,
    )
    return {
        "n_data": len(m_data),
        "n_sig_sm": n_sig_sm,
        "n_bkg_mc": n_bkg_mc,
        "m_h": m_h_val,
        "n_sig": n_sig_val,
        "n_sig_err": n_sig_err,
        "mu_hat": mu_hat,
        "mu_hat_err": mu_hat_err,
    }

We now take a look at the Higgs mass obtained from the cut-based analysis.

In [ ]:
red_mc = data_h[r"Background $Z,t\bar{t},t\bar{t}+V,VVV$"]
zz_mc = data_h[r"Background $ZZ^{*}$"]
sig_mc = data_h[r"Signal ($m_H$ = 125 GeV)"]
bkg_mc = ak.concatenate([red_mc, zz_mc], axis=0)

m_red_raw = np.asarray(ak.to_numpy(red_mc["inv_mass"]))
w_red_raw = np.asarray(ak.to_numpy(red_mc["Weight"]))
m_zz_raw = np.asarray(ak.to_numpy(zz_mc["inv_mass"]))
w_zz_raw = np.asarray(ak.to_numpy(zz_mc["Weight"]))
m_sig_raw = np.asarray(ak.to_numpy(sig_mc["inv_mass"]))
w_sig_raw = np.asarray(ak.to_numpy(sig_mc["Weight"]))
m_bkg_raw = np.asarray(ak.to_numpy(bkg_mc["inv_mass"]))
w_bkg_raw = np.asarray(ak.to_numpy(bkg_mc["Weight"]))
m_data_raw = np.asarray(ak.to_numpy(data_h["Data"]["inv_mass"]))

# Stacked MC vs Data before the BDT cut
plot_mc_vs_data_4l(
    m_red_raw, w_red_raw, m_zz_raw, w_zz_raw, m_sig_raw, w_sig_raw, m_data_raw,
    title=r"Cut-based selection only (before BDT): $H \to ZZ^* \to 4\ell$",
)

# Signal region yields (120-130 GeV) before the BDT cut
sr_sig_pre = (m_sig_raw >= 120.0) & (m_sig_raw < 130.0)
sr_bkg_pre = (m_bkg_raw >= 120.0) & (m_bkg_raw < 130.0)
s_pre_sr = float(np.sum(w_sig_raw[sr_sig_pre]))
b_pre_sr = float(np.sum(w_bkg_raw[sr_bkg_pre]))
print(f"Pre-BDT in [120, 130] GeV: S = {s_pre_sr:.2f}, B = {b_pre_sr:.2f}, "
      f"S/B = {s_pre_sr / b_pre_sr:.2f}, S/sqrt(B) = {s_pre_sr / np.sqrt(b_pre_sr):.2f}")

# Unbinned fit on the cut-based [105, 160] GeV window
pre_win_sig = (m_sig_raw >= h_fit_low) & (m_sig_raw <= h_fit_high)
pre_win_bkg = (m_bkg_raw >= h_fit_low) & (m_bkg_raw <= h_fit_high)
pre_win_dat = (m_data_raw >= h_fit_low) & (m_data_raw <= h_fit_high)

res_pre_bdt = fit_higgs_zfit(
    m_sig_raw[pre_win_sig], w_sig_raw[pre_win_sig],
    m_bkg_raw[pre_win_bkg], w_bkg_raw[pre_win_bkg],
    m_data_raw[pre_win_dat],
    prefix="pre",
    title=r"Unbinned Extended ML Fit (Cut-Based Only, Before BDT): $H \to 4\ell$",
)

### *Apply the 4 BDT selection*

Now we train the same `GradientBoostingClassifier` on kinematic features (without $m_{4\ell}$ itself, to avoid sculpting or biasing the fit variable), optimize the threshold on validation $S/\sqrt{B}$, and see how the spectrum and yields change after the BDT cut.

In [ ]:
# We use the same 15 features as the previous times, including lepton pT, eta, delta_R and MET
def make_bdt_features(sample):
    order = ak.argsort(sample["lep_pt"], axis=1, ascending=False)
    pt = ak.to_numpy(ak.to_regular(sample["lep_pt"][order], axis=1))
    eta = ak.to_numpy(ak.to_regular(sample["lep_eta"][order], axis=1))
    phi = ak.to_numpy(ak.to_regular(sample["lep_phi"][order], axis=1))

    delta_phi = np.abs(np.arctan2(
        np.sin(phi[:, :, None] - phi[:, None, :]),
        np.cos(phi[:, :, None] - phi[:, None, :]),
    ))
    delta_eta = eta[:, :, None] - eta[:, None, :]
    delta_r = np.sqrt(delta_eta**2 + delta_phi**2)
    pair_delta_r = np.column_stack([delta_r[:, i, j] for i, j in combinations(range(4), 2)])
    met = np.asarray(ak.to_numpy(sample["met"]))
    return np.column_stack((pt, eta, pair_delta_r, met))

In [ ]:
# We construct the features and split the data
X_sig, X_bkg = make_bdt_features(sig_mc), make_bdt_features(bkg_mc)
X_all = np.concatenate((X_sig, X_bkg))
y_all = np.concatenate((np.ones(len(X_sig), dtype=int), np.zeros(len(X_bkg), dtype=int)))
w_all = np.concatenate((w_sig_raw, w_bkg_raw))
m_all = np.concatenate((m_sig_raw, m_bkg_raw))

X_tv, X_test_h, y_tv, y_test_h, w_tv, w_test_h, m_tv, m_test_h = train_test_split(
    X_all, y_all, w_all, m_all, test_size=0.2, stratify=y_all, random_state=42
)
X_tr, X_val_h, y_tr, y_val_h, w_tr, w_val_h, m_tr, m_val_h = train_test_split(
    X_tv, y_tv, w_tv, m_tv, test_size=0.25, stratify=y_tv, random_state=42
)

train_w = np.where(y_tr == 1, 0.5 / np.count_nonzero(y_tr == 1), 0.5 / np.count_nonzero(y_tr == 0))

In [ ]:
# We define our BDT model and train it
bdt = GradientBoostingClassifier(n_estimators=200, 
                                 learning_rate=0.05, 
                                 max_depth=2, 
                                 random_state=42)
bdt.fit(X_tr, y_tr, sample_weight=train_w)

In [ ]:
# We validate our model
val_scores = bdt.predict_proba(X_val_h)[:, 1]
val_scale = {lbl: np.count_nonzero(y_val_h == lbl) / np.count_nonzero(y_all == lbl) for lbl in (0, 1)}
w_val_scaled = w_val_h / np.asarray([val_scale[lbl] for lbl in y_val_h])

grid = np.linspace(0.0, 0.95, 40)
s_yields = np.array([np.sum(w_val_scaled[(y_val_h == 1) & (val_scores >= t)]) for t in grid])
b_yields = np.array([np.sum(w_val_scaled[(y_val_h == 0) & (val_scores >= t)]) for t in grid])
fom = np.divide(s_yields, np.sqrt(b_yields), out=np.full_like(s_yields, np.nan), where=(s_yields > 0) & (b_yields > 0))
bdt_cut = float(grid[np.nanargmax(fom)])

sig_scores = bdt.predict_proba(X_sig)[:, 1]
red_scores = bdt.predict_proba(make_bdt_features(red_mc))[:, 1]
zz_scores = bdt.predict_proba(make_bdt_features(zz_mc))[:, 1]
bkg_scores = bdt.predict_proba(X_bkg)[:, 1]
data_4l_scores = bdt.predict_proba(make_bdt_features(data_h["Data"]))[:, 1]

In [ ]:
# We take a look at the 4 lepton invariant mass spectrum
plot_mc_vs_data_4l(
    m_red_raw[red_scores >= bdt_cut], w_red_raw[red_scores >= bdt_cut],
    m_zz_raw[zz_scores >= bdt_cut], w_zz_raw[zz_scores >= bdt_cut],
    m_sig_raw[sig_scores >= bdt_cut], w_sig_raw[sig_scores >= bdt_cut],
    m_data_raw[data_4l_scores >= bdt_cut],
    title=rf"After BDT selection ($\mathrm{{score}} \geq {bdt_cut:.2f}$): $H \to ZZ^* \to 4\ell$",
)

In [ ]:
# We take a look at how events survived the BDT classification
sig_mask = (sig_scores >= bdt_cut) & pre_win_sig
bkg_mask = (bkg_scores >= bdt_cut) & pre_win_bkg
data_mask = (data_4l_scores >= bdt_cut) & pre_win_dat

m4l_sig_mc, w4l_sig_mc = m_sig_raw[sig_mask], w_sig_raw[sig_mask]
m4l_bkg_mc, w4l_bkg_mc = m_bkg_raw[bkg_mask], w_bkg_raw[bkg_mask]
m4l_data = m_data_raw[data_mask]

sr_sig_post = (m4l_sig_mc >= 120.0) & (m4l_sig_mc < 130.0)
sr_bkg_post = (m4l_bkg_mc >= 120.0) & (m4l_bkg_mc < 130.0)
s_post_sr = float(np.sum(w4l_sig_mc[sr_sig_post]))
b_post_sr = float(np.sum(w4l_bkg_mc[sr_bkg_post]))
print(f"BDT cut: {bdt_cut:.2f}")
print(f"Post-BDT in [120, 130] GeV: S = {s_post_sr:.2f}, B = {b_post_sr:.2f}, "
      f"S/B = {s_post_sr / b_post_sr:.2f}, S/sqrt(B) = {s_post_sr / np.sqrt(b_post_sr):.2f}")
print(f"Window [{h_fit_low:.0f}, {h_fit_high:.0f}] GeV -> Data: {len(m4l_data)} events | "
      f"Expected SM signal: {np.sum(w4l_sig_mc):.2f} | Expected bkg: {np.sum(w4l_bkg_mc):.2f}")

### *Why `curve_fit` breaks down in the sparse regime*

After the BDT cut removes most of the background, bins in `105–160 GeV` often contain `0, 1, 2` events. Let's see what happens if we try a binned `scipy.optimize.curve_fit` (`Gaussian + Exponential`) on `m4l_data`.

In [ ]:
centers_h = 0.5 * (bins_h[1:] + bins_h[:-1])
counts_h, _ = np.histogram(m4l_data, bins=bins_h)
err_h = np.sqrt(np.maximum(counts_h, 1.0))

# We define a similar model to what we have above
def h_binned_model(x, n_s, mu, sigma, n_b, lam):
    bw = bins_h[1] - bins_h[0]
    sig = n_s * np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (np.sqrt(2 * np.pi) * sigma)
    if np.isclose(lam, 0.0):
        norm_exp = h_fit_high - h_fit_low
    else:
        norm_exp = (np.exp(lam * h_fit_high) - np.exp(lam * h_fit_low)) / lam
    bkg = n_b * np.exp(lam * x) / norm_exp
    return (sig + bkg) * bw

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(centers_h, counts_h, yerr=np.sqrt(counts_h), fmt="ko", markersize=4, label="Data (post-BDT)")

# We fit the data using curve_fit
popt_h, pcov_h = opt.curve_fit(
    h_binned_model,
    centers_h,
    counts_h,
    p0=[float(np.sum(w4l_sig_mc)), 125.0, 2.0, float(np.sum(w4l_bkg_mc)), -0.02],
    sigma=err_h,
    absolute_sigma=True,
    maxfev=5000,
)
perr_h = np.sqrt(np.diag(pcov_h))
pred_h = h_binned_model(centers_h, *popt_h)
chi2_ndf_h = np.sum(((counts_h - pred_h) / err_h) ** 2) / (len(centers_h) - len(popt_h))

names = ["N_sig", "m_H [GeV]", "sigma_H [GeV]", "N_bkg", "lambda [GeV^-1]"]
print(f"curve_fit results (chi2/ndf = {chi2_ndf_h:.2f}):")
for name, val, err in zip(names, popt_h, perr_h):
    print(f"  {name:<16} = {val:8.3f} +/- {err:8.3f}")

x_fine = np.linspace(h_fit_low, h_fit_high, 400)
ax.plot(x_fine, h_binned_model(x_fine, *popt_h), color="crimson", lw=2, label="curve_fit (biased)")


ax.set_xlabel(r"$m_{4\ell}$ [GeV]")
ax.set_ylabel(f"Events / {(bins_h[1] - bins_h[0]):.1f} GeV")
ax.set_ylim(bottom=0)
ax.legend()
plt.show()

### *Post-BDT unbinned extended likelihood fit & comparison with cut-based*

Now let's run the exact same `fit_higgs_zfit` procedure on the **BDT-selected** sample and compare both the `120–130 GeV` signal-to-background metrics and the fitted $\hat{\mu} \pm \Delta\hat{\mu}$ before vs. after the BDT!


In [ ]:
res_post_bdt = fit_higgs_zfit(
    m4l_sig_mc, w4l_sig_mc,
    m4l_bkg_mc, w4l_bkg_mc,
    m4l_data,
    prefix="post",
    title=r"Unbinned Extended ML Fit (After BDT): $H \to ZZ^* \to 4\ell$",
)

print("\n=== Pre-BDT (Cut-Based) vs. Post-BDT Comparison in [105, 160] GeV ===")
print(f"{'Metric':<28} {'Cut-Based (Pre-BDT)':>22} {'BDT-Selected (Post-BDT)':>25}")
print("-" * 77)
print(f"{'Data events':<28} {res_pre_bdt['n_data']:>22d} {res_post_bdt['n_data']:>25d}")
print(f"{'Expected SM signal':<28} {res_pre_bdt['n_sig_sm']:>22.2f} {res_post_bdt['n_sig_sm']:>25.2f}")
print(f"{'Expected MC bkg':<28} {res_pre_bdt['n_bkg_mc']:>22.2f} {res_post_bdt['n_bkg_mc']:>25.2f}")
print(f"{'Fitted m_H [GeV]':<28} {res_pre_bdt['m_h']:>22.2f} {res_post_bdt['m_h']:>25.2f}")
pre_nsig_str = f"{res_pre_bdt['n_sig']:.1f} +/- {res_pre_bdt['n_sig_err']:.1f}"
post_nsig_str = f"{res_post_bdt['n_sig']:.1f} +/- {res_post_bdt['n_sig_err']:.1f}"
print(f"{'Fitted N_sig':<28} {pre_nsig_str:>22} {post_nsig_str:>25}")
pre_mu_str = f"{res_pre_bdt['mu_hat']:.2f} +/- {res_pre_bdt['mu_hat_err']:.2f}"
post_mu_str = f"{res_post_bdt['mu_hat']:.2f} +/- {res_post_bdt['mu_hat_err']:.2f}"
print(f"{'Signal strength mu_hat':<28} {pre_mu_str:>22} {post_mu_str:>25}")


### *What to do now?*

Here are a few extensions to explore:
* **Asymmetric detector tails:** Replace `zfit.pdf.Gauss` with `zfit.pdf.CrystalBall` for the $Z \to \mu^+\mu^-$ or $H \to 4\ell$ signal shape (final-state radiation creates a low-mass tail). Does it improve the pulls?
* **Constrained nuisance parameters instead of hard-fixing:** In Part 2, keep `sigma_h` and `lam_h` floating (`floating = True`), and pass a `zfit.constraint.GaussianConstraint` using the MC fit uncertainties. How much does $\Delta m_H$ or $\Delta N_{\text{sig}}$ grow?
* **Add the $Z \to 4\ell$ peak:** Widen the Part 2 mass window to `80–170 GeV` and add a third `zfit.pdf.Voigt` component around $91.2\text{ GeV}$ for $Z \to 4\ell$ decays.
* **Preview discovery significance:** Fit a background-only model ($N_{\text{sig}} = 0$) to `m4l_data` and compute $\sqrt{2 \, (\text{NLL}_{\text{bkg-only}} - \text{NLL}_{\text{sig+bkg}})}$. Compare that quick likelihood-ratio estimate to the $S/\sqrt{B}$ metric from Lab 4!
